<a href="https://colab.research.google.com/github/Thanu10ekoon/AI_Learning/blob/main/nb/Gemma3_(4B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [3]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 6,           # Larger = higher accuracy, but might overfit
    lora_alpha = 6,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [6]:
from datasets import load_dataset
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split = "train")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [7]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [8]:
dataset[100]

{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.",
 'input': 'Why does husband have saggy balls that hang low and can t seem to last long when having sex? Seems like he gets off within 5 minutes and he claims he can t or don t know how to last longer, even though I ve told him some techniques to try but still says he can t and/or don t know how. His balls aren t full and firm either like my ex s were. Also have been trying to conceive for a long time and have not had any success, what is going on??? How to know if he is even ejaculating sperm and not just semen/fluid, what is the difference. Healthy sperm should be white or more yellowish??? Really need some help please......',
 'output': 'HelloThanks for query. The scrotal sac differs from person to person and has no relation to conception. The problem of premature ejaculation and less libido are most of the time mind related and due to what is called as performance anxiety.

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [9]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        # Build a proper chat structure for Gemma
        messages = [
            {"role": "user", "content": f"{inst}\n\n{inp}"},
            {"role": "assistant", "content": out},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix("<bos>")

        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [10]:
dataset[100]["text"]

"<start_of_turn>user\nIf you are a doctor, please answer the medical questions based on the patient's description.\n\nWhy does husband have saggy balls that hang low and can t seem to last long when having sex? Seems like he gets off within 5 minutes and he claims he can t or don t know how to last longer, even though I ve told him some techniques to try but still says he can t and/or don t know how. His balls aren t full and firm either like my ex s were. Also have been trying to conceive for a long time and have not had any success, what is going on??? How to know if he is even ejaculating sperm and not just semen/fluid, what is the difference. Healthy sperm should be white or more yellowish??? Really need some help please......<end_of_turn>\n<start_of_turn>model\nHelloThanks for query. The scrotal sac differs from person to person and has no relation to conception. The problem of premature ejaculation and less libido are most of the time mind related and due to what is called as per

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [11]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [12]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

Unsloth: Removed 1 out of 112165 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [13]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><start_of_turn>user\nIf you are a doctor, please answer the medical questions based on the patient's description.\n\nWhy does husband have saggy balls that hang low and can t seem to last long when having sex? Seems like he gets off within 5 minutes and he claims he can t or don t know how to last longer, even though I ve told him some techniques to try but still says he can t and/or don t know how. His balls aren t full and firm either like my ex s were. Also have been trying to conceive for a long time and have not had any success, what is going on??? How to know if he is even ejaculating sperm and not just semen/fluid, what is the difference. Healthy sperm should be white or more yellowish??? Really need some help please......<end_of_turn>\n<start_of_turn>model\nHelloThanks for query. The scrotal sac differs from person to person and has no relation to conception. The problem of premature ejaculation and less libido are most of the time mind related and due to what is called a

Now let's print the masked out example - you should see only the answer is present:

In [14]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                           HelloThanks for query. The scrotal sac differs from person to person and has no relation to conception. The problem of premature ejaculation and less libido are most of the time mind related and due to what is called as performance anxiety. Following measure will help you to boost up his confidence and getting good erection. And delay ejaculation.1) Practice regular exercise for 45 minutes followed by meditation for 1/2 an hour in the morning.2) Take high protein diet rich in vegetables and fruits and vitamin A, C,D, E.and Zinc3)Take anti oxidants like Almonds 5-6 every day.4) Avoid alcohol and smoking. He may take Sidewall (Viagra)as on demand to have sustainable erection to have enjoyable sex. As regards conception get his semen analysis done to rule out any abnormality. Chat Doctor.<end_of_turn> '

In [15]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.381 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 112,164 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,175,936 of 4,311,255,408 (0.26% trained)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn'

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,4.903300
2,5.200400
3,4.774900
4,4.739000
5,4.131200
6,3.701900
7,3.490200
8,3.159800
9,3.003000
10,3.148300


In [17]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

240.8268 seconds used for training.
4.01 minutes used for training.
Peak reserved memory = 5.508 GB.
Peak reserved memory for training = 1.127 GB.
Peak reserved memory % of max memory = 37.822 %.
Peak reserved memory for training % of max memory = 7.739 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [18]:
from unsloth.chat_templates import get_chat_template

# Ensure correct template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

# Medical-style query
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "I have had a fever and headache for two days. What could be the reason and what should I do?"
            }
        ]
    }
]

# Tokenize with generation prompt
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

# Generate response
outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 200,  # longer for detailed medical answers
    temperature = 0.7,     # more stable/less random
    top_p = 0.9,
    top_k = 50,
)

# Decode output
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0])

user
I have had a fever and headache for two days. What could be the reason and what should I do?
model
Hi I am Dr. Chat, an AI Chatbot. Your symptoms of fever and headache for two days can be due to a variety of causes like common cold, influenza, or other viral infections. You may be suffering from a sinus infection. You may have had a COVID infection, so a COVID test is recommended. Your headache is most likely due to fever, headache, and fever is common in COVID infection. The best course of action is to consult with a doctor to get an accurate diagnosis and appropriate treatment. Thanks and good luck.


 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [19]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "You are a careful medical assistant. Do not give definitive diagnoses. "
                    "Suggest possible causes and advise when to see a doctor.\n\n"
                    "Patient: I have had a fever and headache for two days. What should I do?"
                ),
            }
        ]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer

_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 200,   # longer for medical responses
    temperature = 0.7,      # safer, less random
    top_p = 0.9,
    top_k = 50,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
)

Hello, I understand you are concerned about fever and headache. They may be caused by infection, which may be bacterial or viral. These are common symptoms of viral infections such as influenza. Headache may be caused by migraine or tension headache. Please consult a doctor for further evaluation. Hope that helps.<end_of_turn>


--------------------------------------------------------------------------------

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [20]:
model.save_pretrained("gemma_3_lora")  # Local saving
tokenizer.save_pretrained("gemma_3_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_3_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_3_lora", token = "YOUR_HF_TOKEN") # Online saving

['gemma_3_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [21]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_3_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-3?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
)

from transformers import TextStreamer
_ = model.generate(
    **inputs.to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Gemma-3 is a family of open-source large language models (LLMs) developed by the team at Stability AI. They are based on the same foundational architectures that underpin other LLMs like Llama and are significantly smaller which is an advantage in terms of memory and energy usage. They are freely available for commercial


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [22]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [23]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [24]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_3_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [25]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_3_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Save Model

In [26]:
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

In [27]:
from huggingface_hub import login
login(token=hf_token)

In [28]:
model.push_to_hub("Thanu10/Arogya_Finetuned", token=hf_token)

README.md:   0%|          | 0.00/576 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  565kB / 44.8MB            

Saved model to https://huggingface.co/Thanu10/Arogya_Finetuned


In [30]:
if True: # Changed from False to True to upload the merged finetune
    model.push_to_hub_merged(
        "Thanu10/Arogya_finetuned_merged", tokenizer,
        token=hf_token # Insert your HF write token here
    )

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ed_merged/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...ned_merged/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

...

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:57<?, ?it/s]


KeyboardInterrupt: 